# 📊 Finance & Risk Analytics Dashboard
### A Comprehensive Portfolio Analysis & ML Prediction System

---

| Component | Details |
|-----------|---------|
| **Author** | Madhura |
| **Tech Stack** | Python, Pandas, NumPy, Scikit-learn, Plotly, yfinance |
| **Domains** | Quantitative Finance · Risk Management · Machine Learning |
| **Project Type** | End-to-end Data Analytics · Resume Showcase |

---

## 🗂️ Project Sections

1. [Environment Setup & Data Collection](#1)
2. [Portfolio Construction & Return Analysis](#2)
3. [Risk Metrics — VaR, CVaR, Sharpe, Sortino, Beta, Drawdown](#3)
4. [Correlation & Covariance Analysis](#4)
5. [ML Price Prediction — Random Forest + LSTM-style Features](#5)
6. [Monte Carlo Portfolio Simulation](#6)
7. [Interactive Dashboard — Plotly](#7)


In [1]:
import sys
!{sys.executable} -m pip install numpy pandas scipy scikit-learn yfinance plotly matplotlib seaborn ipywidgets

In [2]:
# ── Install dependencies (uncomment if running fresh) ──────────────────────────
# !pip install yfinance pandas numpy matplotlib seaborn plotly scikit-learn scipy

import warnings
warnings.filterwarnings('ignore')

# Core
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Finance data
import yfinance as yf

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ML
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# Stats
from scipy import stats
from scipy.optimize import minimize

# Notebook display
from IPython.display import display, HTML
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print("✅  All libraries loaded successfully")
print(f"📅  Analysis Date: {datetime.today().strftime('%B %d, %Y')}")


✅  All libraries loaded successfully
📅  Analysis Date: March 16, 2026


## 1. Environment Setup & Data Collection <a id='1'></a>

In [24]:
# ── Portfolio Configuration ─────────────────────────────────────────────────────
TICKERS   = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'JPM', 'GS']
BENCHMARK = 'SPY'   # S&P 500 ETF as benchmark

START_DATE = '2020-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

# Equal weight portfolio (can be customised)
WEIGHTS = np.array([0.20, 0.20, 0.15, 0.15, 0.10, 0.10, 0.05, 0.05])
assert abs(WEIGHTS.sum() - 1.0) < 1e-9, "Weights must sum to 1"

RISK_FREE_RATE = 0.05   # 5% annualised (approx US T-bill)
TRADING_DAYS   = 252
CONF_LEVEL     = 0.95   # 95% confidence for VaR

print(f"📌  Tickers  : {TICKERS}")
print(f"📌  Weights  : {dict(zip(TICKERS, WEIGHTS))}")
print(f"📌  Period   : {START_DATE} → {END_DATE}")


📌  Tickers  : ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'JPM', 'GS']
📌  Weights  : {'AAPL': np.float64(0.2), 'MSFT': np.float64(0.2), 'GOOGL': np.float64(0.15), 'AMZN': np.float64(0.15), 'TSLA': np.float64(0.1), 'NVDA': np.float64(0.1), 'JPM': np.float64(0.05), 'GS': np.float64(0.05)}
📌  Period   : 2020-01-01 → 2026-03-16


In [25]:
# ── Download Price Data ─────────────────────────────────────────────────────────
all_tickers = TICKERS + [BENCHMARK]

raw = yf.download(all_tickers, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
prices = raw['Close'].dropna(how='all')

# Separate portfolio vs benchmark
port_prices = prices[TICKERS].copy()
bench_prices = prices[BENCHMARK].copy()

print(f"✅  Downloaded {len(port_prices)} trading days × {len(TICKERS)} stocks")
print(f"📊  Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
display(port_prices.tail())


✅  Downloaded 1557 trading days × 8 stocks
📊  Date range: 2020-01-02 → 2026-03-13


Ticker,AAPL,MSFT,GOOGL,AMZN,TSLA,NVDA,JPM,GS
Date,,,,,,,,
2026-03-09,259.8800,409.4100,306.3600,213.4900,398.6800,182.6401,289.9200,832.0300
2026-03-10,260.8300,405.7600,307.0400,214.3300,399.2400,184.7600,288.7300,833.8100
2026-03-11,260.8100,404.8800,308.7000,212.6500,407.8200,186.0300,287.5200,823.7600
2026-03-12,255.7600,401.8600,303.5500,209.5300,395.0100,183.1400,282.8900,787.5200
2026-03-13,250.1200,395.5500,302.2800,207.6700,391.2000,180.2500,283.4400,782.2100


## 2. Portfolio Construction & Return Analysis <a id='2'></a>

In [26]:
# ── Compute Returns ─────────────────────────────────────────────────────────────
daily_returns   = port_prices.pct_change().dropna()
bench_returns   = bench_prices.pct_change().dropna()

# Align dates
daily_returns, bench_returns = daily_returns.align(bench_returns, join='inner', axis=0)

# Weighted portfolio daily return
port_daily_ret  = daily_returns.dot(WEIGHTS)

# Cumulative returns
cum_port  = (1 + port_daily_ret).cumprod()
cum_bench = (1 + bench_returns).cumprod()
cum_each  = (1 + daily_returns).cumprod()

print("📈  Cumulative Portfolio Return : {:.1f}%".format((cum_port.iloc[-1] - 1) * 100))
print("📈  Cumulative Benchmark Return : {:.1f}%".format((cum_bench.iloc[-1] - 1) * 100))


📈  Cumulative Portfolio Return : 450.6%
📈  Cumulative Benchmark Return : 122.5%


In [27]:
# ── Cumulative Returns Chart (Plotly) ───────────────────────────────────────────
fig = go.Figure()

palette = px.colors.qualitative.Plotly
for i, ticker in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=cum_each.index, y=cum_each[ticker],
        name=ticker, line=dict(width=1.2, color=palette[i % len(palette)]),
        opacity=0.6, visible='legendonly'
    ))

fig.add_trace(go.Scatter(
    x=cum_port.index, y=cum_port,
    name='📦 Portfolio', line=dict(width=3, color='#2ecc71'),
))
fig.add_trace(go.Scatter(
    x=cum_bench.index, y=cum_bench,
    name='📊 SPY Benchmark', line=dict(width=2, color='#e74c3c', dash='dash'),
))

fig.update_layout(
    title='Cumulative Returns: Portfolio vs Benchmark (toggle individual stocks)',
    xaxis_title='Date', yaxis_title='Growth of $1',
    template='plotly_dark', height=500,
    legend=dict(orientation='v', x=1.01),
    hovermode='x unified'
)
fig.show()


In [28]:
# ── Annual Returns Summary ──────────────────────────────────────────────────────
def annual_return(ret_series):
    return (1 + ret_series).resample('YE').prod() - 1

ann_port  = annual_return(port_daily_ret) * 100
ann_bench = annual_return(bench_returns)  * 100

ann_df = pd.DataFrame({
    'Portfolio (%)': ann_port.values,
    'Benchmark SPY (%)': ann_bench.values,
    'Alpha (%)': (ann_port.values - ann_bench.values)
}, index=ann_port.index.year)
ann_df.index.name = 'Year'

fig = go.Figure()
fig.add_trace(go.Bar(x=ann_df.index.astype(str), y=ann_df['Portfolio (%)'],
                     name='Portfolio', marker_color='#2ecc71'))
fig.add_trace(go.Bar(x=ann_df.index.astype(str), y=ann_df['Benchmark SPY (%)'],
                     name='SPY', marker_color='#e74c3c'))
fig.add_trace(go.Bar(x=ann_df.index.astype(str), y=ann_df['Alpha (%)'],
                     name='Alpha', marker_color='#3498db'))
fig.update_layout(barmode='group', title='Annual Returns by Year',
                  template='plotly_dark', height=400, yaxis_ticksuffix='%')
fig.show()
display(ann_df.style.format('{:.2f}%').background_gradient(cmap='RdYlGn', axis=None))


,Portfolio (%),Benchmark SPY (%),Alpha (%)
Year,,,
2020,90.21%,17.24%,72.97%
2021,49.43%,28.73%,20.70%
2022,-36.70%,-18.18%,-18.53%
2023,77.16%,26.18%,50.99%
2024,49.60%,24.89%,24.72%
2025,27.63%,17.72%,9.91%
2026,-9.52%,-2.88%,-6.65%


## 3. Risk Metrics <a id='3'></a>

> **VaR (Value at Risk)**: Maximum expected loss at a given confidence level over a time horizon.  
> **CVaR (Conditional VaR / Expected Shortfall)**: Average loss *beyond* the VaR threshold — captures tail risk.  
> **Sharpe Ratio**: Risk-adjusted return (excess return per unit of total risk).  
> **Sortino Ratio**: Like Sharpe but penalises only *downside* volatility.  
> **Beta**: Portfolio sensitivity to market movements.  
> **Maximum Drawdown**: Largest peak-to-trough decline.


In [29]:
# ── Core Risk Metrics Functions ─────────────────────────────────────────────────
def historical_var(returns, confidence=0.95):
    """Historical simulation VaR."""
    return -np.percentile(returns, (1 - confidence) * 100)

def parametric_var(returns, confidence=0.95):
    """Variance-Covariance (parametric) VaR."""
    mu, sigma = returns.mean(), returns.std()
    z = stats.norm.ppf(1 - confidence)
    return -(mu + z * sigma)

def conditional_var(returns, confidence=0.95):
    """Expected Shortfall / CVaR."""
    var = historical_var(returns, confidence)
    return -returns[returns <= -var].mean()

def sharpe_ratio(returns, rf=RISK_FREE_RATE, periods=TRADING_DAYS):
    """Annualised Sharpe Ratio."""
    excess = returns - rf / periods
    return np.sqrt(periods) * excess.mean() / returns.std()

def sortino_ratio(returns, rf=RISK_FREE_RATE, periods=TRADING_DAYS):
    """Annualised Sortino Ratio."""
    excess = returns - rf / periods
    downside = returns[returns < 0].std()
    return np.sqrt(periods) * excess.mean() / downside

def max_drawdown(cum_returns):
    """Maximum peak-to-trough drawdown."""
    rolling_max = cum_returns.cummax()
    drawdown    = (cum_returns - rolling_max) / rolling_max
    return drawdown.min()

def portfolio_beta(port_ret, bench_ret):
    """Portfolio Beta vs benchmark."""
    cov_matrix = np.cov(port_ret, bench_ret)
    return cov_matrix[0, 1] / cov_matrix[1, 1]

def calmar_ratio(returns, cum_ret, periods=TRADING_DAYS):
    """Annualised return divided by max drawdown."""
    ann_ret = returns.mean() * periods
    mdd     = abs(max_drawdown(cum_ret))
    return ann_ret / mdd if mdd != 0 else np.nan

print("✅  Risk metric functions defined")


✅  Risk metric functions defined


In [30]:
# ── Compute All Metrics for Portfolio & Each Stock ──────────────────────────────
metrics = {}

# Portfolio-level
metrics['Portfolio'] = {
    'Ann. Return (%)':     port_daily_ret.mean() * TRADING_DAYS * 100,
    'Ann. Volatility (%)': port_daily_ret.std()  * np.sqrt(TRADING_DAYS) * 100,
    'Hist. VaR 95% (%)':   historical_var(port_daily_ret) * 100,
    'Param. VaR 95% (%)':  parametric_var(port_daily_ret) * 100,
    'CVaR 95% (%)':        conditional_var(port_daily_ret) * 100,
    'Sharpe Ratio':        sharpe_ratio(port_daily_ret),
    'Sortino Ratio':       sortino_ratio(port_daily_ret),
    'Beta':                portfolio_beta(port_daily_ret.values, bench_returns.values),
    'Max Drawdown (%)':    max_drawdown(cum_port) * 100,
    'Calmar Ratio':        calmar_ratio(port_daily_ret, cum_port),
}

# Per-stock metrics
for ticker in TICKERS:
    r = daily_returns[ticker]
    c = cum_each[ticker]
    metrics[ticker] = {
        'Ann. Return (%)':     r.mean() * TRADING_DAYS * 100,
        'Ann. Volatility (%)': r.std()  * np.sqrt(TRADING_DAYS) * 100,
        'Hist. VaR 95% (%)':   historical_var(r) * 100,
        'Param. VaR 95% (%)':  parametric_var(r) * 100,
        'CVaR 95% (%)':        conditional_var(r) * 100,
        'Sharpe Ratio':        sharpe_ratio(r),
        'Sortino Ratio':       sortino_ratio(r),
        'Beta':                portfolio_beta(r.values, bench_returns.values),
        'Max Drawdown (%)':    max_drawdown(c) * 100,
        'Calmar Ratio':        calmar_ratio(r, c),
    }

metrics_df = pd.DataFrame(metrics).T
display(metrics_df.style
    .format('{:.3f}')
    .background_gradient(cmap='RdYlGn', subset=['Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio'])
    .background_gradient(cmap='RdYlGn_r', subset=['Hist. VaR 95% (%)', 'CVaR 95% (%)', 'Max Drawdown (%)'])
    .set_caption("📊 Risk Metrics Summary — Portfolio & Individual Holdings"))


,Ann. Return (%),Ann. Volatility (%),Hist. VaR 95% (%),Param. VaR 95% (%),CVaR 95% (%),Sharpe Ratio,Sortino Ratio,Beta,Max Drawdown (%),Calmar Ratio
Portfolio,31.859,29.028,2.851,2.881,4.222,0.925,1.246,1.273,-40.685,0.783
AAPL,25.075,31.646,3.077,3.180,4.460,0.634,0.895,1.201,-33.361,0.752
MSFT,19.899,29.735,2.789,3.002,4.220,0.501,0.689,1.143,-37.148,0.536
GOOGL,29.381,32.193,3.074,3.219,4.562,0.757,1.081,1.131,-44.320,0.663
AMZN,18.997,35.558,3.299,3.609,4.992,0.394,0.578,1.137,-56.145,0.338
TSLA,63.901,65.817,5.922,6.566,8.891,0.895,1.364,1.702,-73.632,0.868
NVDA,69.045,52.733,4.874,5.190,6.957,1.215,1.885,1.801,-66.335,1.041
JPM,18.919,31.239,2.875,3.162,4.468,0.446,0.599,1.075,-43.062,0.439
GS,27.333,32.928,2.830,3.303,4.613,0.678,0.944,1.194,-45.621,0.599


In [31]:
# ── VaR Distribution Plot ────────────────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Portfolio Return Distribution + VaR',
                    'Rolling 30-Day VaR (Historical)'])

x = np.linspace(port_daily_ret.min(), port_daily_ret.max(), 300)
hist_var = historical_var(port_daily_ret)
cvar_val  = conditional_var(port_daily_ret)

# KDE + VaR lines
fig.add_trace(go.Histogram(x=port_daily_ret, nbinsx=80, histnorm='probability density',
    name='Daily Returns', marker_color='#3498db', opacity=0.6), row=1, col=1)
mu, sigma = port_daily_ret.mean(), port_daily_ret.std()
fig.add_trace(go.Scatter(x=x, y=stats.norm.pdf(x, mu, sigma),
    name='Normal Fit', line=dict(color='white', width=2)), row=1, col=1)
fig.add_vline(x=-hist_var, line_color='#e74c3c', line_dash='dash',
              annotation_text=f'VaR {hist_var*100:.2f}%', row=1, col=1)
fig.add_vline(x=-cvar_val, line_color='#e67e22', line_dash='dot',
              annotation_text=f'CVaR {cvar_val*100:.2f}%', row=1, col=1)

# Rolling VaR
window = 30
rolling_var = port_daily_ret.rolling(window).apply(
    lambda x: historical_var(x), raw=False) * 100
fig.add_trace(go.Scatter(x=rolling_var.index, y=rolling_var,
    name=f'{window}d Rolling VaR', line=dict(color='#e74c3c', width=1.5)), row=1, col=2)

fig.update_layout(template='plotly_dark', height=420,
    title='Value at Risk Analysis', hovermode='x unified')
fig.show()


In [32]:
# ── Drawdown Chart ───────────────────────────────────────────────────────────────
drawdown = (cum_port - cum_port.cummax()) / cum_port.cummax() * 100

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=drawdown.index, y=drawdown, fill='tozeroy',
    name='Drawdown', line=dict(color='#e74c3c', width=1.5),
    fillcolor='rgba(231,76,60,0.25)'
))
fig.add_hline(y=float(max_drawdown(cum_port) * 100),
              line_color='orange', line_dash='dash',
              annotation_text=f"Max DD: {max_drawdown(cum_port)*100:.1f}%")
fig.update_layout(template='plotly_dark', height=380,
    title='Portfolio Drawdown Over Time',
    xaxis_title='Date', yaxis_title='Drawdown (%)', hovermode='x unified')
fig.show()


## 4. Correlation & Covariance Analysis <a id='4'></a>

In [33]:
# ── Correlation Heatmap ──────────────────────────────────────────────────────────
corr_matrix = daily_returns.corr()

fig = px.imshow(
    corr_matrix, text_auto='.2f',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Pairwise Correlation Matrix — Daily Returns',
    aspect='auto'
)
fig.update_layout(template='plotly_dark', height=520,
    coloraxis_colorbar=dict(title='Correlation'))
fig.show()


In [34]:
# ── Rolling Correlation vs Benchmark ────────────────────────────────────────────
window = 60
roll_corr = daily_returns.rolling(window).corr(bench_returns)

fig = go.Figure()
palette = px.colors.qualitative.Plotly
for i, ticker in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=roll_corr.index, y=roll_corr[ticker],
        name=ticker, line=dict(width=1.5, color=palette[i % len(palette)])
    ))

fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(template='plotly_dark', height=420,
    title=f'{window}-Day Rolling Correlation vs SPY',
    xaxis_title='Date', yaxis_title='Correlation', hovermode='x unified')
fig.show()


## 5. ML Price Prediction <a id='5'></a>

Using **Random Forest** and **Gradient Boosting** regressors with rich technical-indicator features — 
a production-style ML pipeline with **TimeSeriesSplit** cross-validation (no data leakage).


In [35]:
# ── Feature Engineering ─────────────────────────────────────────────────────────
def build_features(price_series, ticker='AAPL'):
    """
    Build a feature matrix from OHLCV data using technical indicators.
    Includes lag returns, rolling statistics, momentum, and volatility features.
    """
    df = pd.DataFrame()

    # Raw price & log return
    df['price']     = price_series
    df['log_ret']   = np.log(price_series / price_series.shift(1))

    # Lag returns (1-day, 2-day, 5-day, 10-day)
    for lag in [1, 2, 3, 5, 10, 21]:
        df[f'ret_lag_{lag}'] = df['log_ret'].shift(lag)

    # Rolling statistics
    for w in [5, 10, 21, 63]:
        df[f'rolling_mean_{w}']  = price_series.rolling(w).mean()
        df[f'rolling_std_{w}']   = price_series.rolling(w).std()
        df[f'rolling_skew_{w}']  = df['log_ret'].rolling(w).skew()
        df[f'rolling_kurt_{w}']  = df['log_ret'].rolling(w).kurt()
        df[f'price_vs_ma_{w}']   = price_series / df[f'rolling_mean_{w}'] - 1

    # Momentum (price ratio over windows)
    for w in [5, 21, 63, 126]:
        df[f'momentum_{w}'] = price_series / price_series.shift(w) - 1

    # Volatility regime
    df['realised_vol_21'] = df['log_ret'].rolling(21).std() * np.sqrt(252)
    df['vol_ratio']       = df['realised_vol_21'] / df['log_ret'].rolling(63).std() / np.sqrt(252)

    # RSI (14-period)
    delta = price_series.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = price_series.ewm(span=12, adjust=False).mean()
    ema26 = price_series.ewm(span=26, adjust=False).mean()
    df['macd']        = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']

    # Bollinger Band position
    bb_mid = price_series.rolling(20).mean()
    bb_std = price_series.rolling(20).std()
    df['bb_position'] = (price_series - bb_mid) / (2 * bb_std)

    # Target: next 5-day return (forward-looking — shifted to avoid leakage)
    df['target'] = df['log_ret'].shift(-5).rolling(5).sum()

    df.dropna(inplace=True)
    return df

TARGET_TICKER = 'AAPL'
feat_df = build_features(port_prices[TARGET_TICKER], ticker=TARGET_TICKER)

feature_cols = [c for c in feat_df.columns if c not in ['price', 'log_ret', 'target']]
X = feat_df[feature_cols]
y = feat_df['target']

print(f"✅  Feature matrix: {X.shape[0]} samples × {X.shape[1]} features")
print(f"📌  Predicting: 5-day forward log-return for {TARGET_TICKER}")
X.describe().T[['mean','std','min','max']].head(8)


✅  Feature matrix: 1426 samples × 37 features
📌  Predicting: 5-day forward log-return for AAPL


,mean,std,min,max
ret_lag_1,0.0008,0.0183,-0.0970,0.1426
ret_lag_2,0.0008,0.0183,-0.0970,0.1426
ret_lag_3,0.0008,0.0183,-0.0970,0.1426
ret_lag_5,0.0008,0.0183,-0.0970,0.1426
ret_lag_10,0.0008,0.0183,-0.0970,0.1426
ret_lag_21,0.0009,0.0183,-0.0970,0.1426
rolling_mean_5,175.6721,45.0537,87.6293,282.3338
rolling_std_5,2.6137,1.7921,0.2470,20.1133


In [36]:
# ── TimeSeriesSplit Train / Test Split ───────────────────────────────────────────
TRAIN_FRAC = 0.80
split_idx  = int(len(X) * TRAIN_FRAC)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape} | {X_train.index[0].date()} → {X_train.index[-1].date()}")
print(f"Test : {X_test.shape}  | {X_test.index[0].date()}  → {X_test.index[-1].date()}")


Train: (1140, 37) | 2020-07-02 → 2025-01-14
Test : (286, 37)  | 2025-01-15  → 2026-03-06


In [37]:
# ── Train Models ─────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Random Forest ─────────────────────────────────────────
rf = RandomForestRegressor(
    n_estimators=400, max_depth=8, min_samples_leaf=10,
    max_features=0.5, random_state=42, n_jobs=-1
)
rf.fit(X_train_sc, y_train)
rf_pred = rf.predict(X_test_sc)

# ── Gradient Boosting ─────────────────────────────────────
gb = GradientBoostingRegressor(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=10, random_state=42
)
gb.fit(X_train_sc, y_train)
gb_pred = gb.predict(X_test_sc)

# ── Ensemble (average) ────────────────────────────────────
ens_pred = (rf_pred + gb_pred) / 2

def eval_model(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    ic   = pd.Series(y_true).corr(pd.Series(y_pred))    # Information Coefficient
    print(f"  {name:<22}  MAE={mae:.5f}  RMSE={rmse:.5f}  R²={r2:.4f}  IC={ic:.4f}")

print("📊  Model Evaluation on Hold-Out Test Set:")
eval_model("Random Forest",      y_test, rf_pred)
eval_model("Gradient Boosting",  y_test, gb_pred)
eval_model("Ensemble (avg)",     y_test, ens_pred)


📊  Model Evaluation on Hold-Out Test Set:
  Random Forest           MAE=0.03486  RMSE=0.04702  R²=-0.0128  IC=nan
  Gradient Boosting       MAE=0.03840  RMSE=0.05101  R²=-0.1916  IC=nan
  Ensemble (avg)          MAE=0.03582  RMSE=0.04806  R²=-0.0579  IC=nan


In [38]:
# ── Prediction vs Actual Plot ────────────────────────────────────────────────────
fig = make_subplots(rows=2, cols=1,
    subplot_titles=['Predicted vs Actual 5-Day Forward Return',
                    'Prediction Error Distribution'],
    row_heights=[0.65, 0.35])

dates = X_test.index

fig.add_trace(go.Scatter(x=dates, y=y_test.values, name='Actual',
    line=dict(color='white', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=rf_pred, name='Random Forest',
    line=dict(color='#3498db', width=1.5, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=gb_pred, name='Gradient Boosting',
    line=dict(color='#2ecc71', width=1.5, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=ens_pred, name='Ensemble',
    line=dict(color='#e67e22', width=2.5)), row=1, col=1)

errors = ens_pred - y_test.values
fig.add_trace(go.Histogram(x=errors, nbinsx=60, name='Ensemble Error',
    marker_color='#e67e22', opacity=0.7), row=2, col=1)

fig.update_layout(template='plotly_dark', height=600,
    title=f'ML Price Prediction — {TARGET_TICKER}', hovermode='x unified')
fig.show()


In [39]:
# ── Feature Importance ───────────────────────────────────────────────────────────
imp_df = pd.DataFrame({
    'Feature':    feature_cols,
    'RF Importance':  rf.feature_importances_,
    'GB Importance':  gb.feature_importances_,
}).sort_values('RF Importance', ascending=False).head(20)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Random Forest — Top 20 Features',
                    'Gradient Boosting — Top 20 Features'])

fig.add_trace(go.Bar(
    y=imp_df['Feature'], x=imp_df['RF Importance'],
    orientation='h', marker_color='#3498db', name='RF'), row=1, col=1)

gb_imp = pd.DataFrame({'Feature': feature_cols,
                        'GB Importance': gb.feature_importances_}
).sort_values('GB Importance', ascending=False).head(20)

fig.add_trace(go.Bar(
    y=gb_imp['Feature'], x=gb_imp['GB Importance'],
    orientation='h', marker_color='#2ecc71', name='GB'), row=1, col=2)

fig.update_layout(template='plotly_dark', height=580,
    title='Feature Importance Comparison', showlegend=False)
fig.show()


In [40]:
# ── Directional Accuracy (Trading Signal) ───────────────────────────────────────
signal_actual = np.sign(y_test.values)
signal_pred   = np.sign(ens_pred)

direction_acc = np.mean(signal_actual == signal_pred)
print(f"🎯  Directional Accuracy (Ensemble): {direction_acc:.1%}")
print(f"    (50% = random guess, >55% considered useful in quant finance)")

# Simulate a long/short strategy based on signal
signal_ser    = pd.Series(signal_pred, index=X_test.index)
strategy_ret  = signal_ser.shift(1) * bench_returns.reindex(X_test.index)
cumulative_strat = (1 + strategy_ret.dropna()).cumprod()
cumulative_bh    = (1 + bench_returns.reindex(X_test.index)).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=cumulative_strat.index, y=cumulative_strat,
    name='ML Signal Strategy', line=dict(color='#e67e22', width=2)))
fig.add_trace(go.Scatter(x=cumulative_bh.index, y=cumulative_bh,
    name='Buy & Hold SPY', line=dict(color='#e74c3c', width=2, dash='dash')))
fig.update_layout(template='plotly_dark', height=380,
    title='Simulated Trading Strategy (Test Period)',
    xaxis_title='Date', yaxis_title='Cumulative Return', hovermode='x unified')
fig.show()


🎯  Directional Accuracy (Ensemble): 57.3%
    (50% = random guess, >55% considered useful in quant finance)


## 6. Monte Carlo Portfolio Simulation <a id='6'></a>

Simulating **10,000 possible portfolio paths** over the next year using Cholesky decomposition 
to preserve the correlation structure between assets.


In [41]:
# ── Monte Carlo with Correlated Asset Paths ─────────────────────────────────────
N_SIMULATIONS = 10_000
N_DAYS        = 252
PORTFOLIO_VALUE = 100_000   # ₹ or $ starting value

np.random.seed(42)

mu_vec    = daily_returns.mean().values            # drift vector
cov_mat   = daily_returns.cov().values             # covariance matrix
chol      = np.linalg.cholesky(cov_mat)            # Cholesky decomposition

final_values = np.zeros(N_SIMULATIONS)
all_paths    = np.zeros((N_SIMULATIONS, N_DAYS))

for sim in range(N_SIMULATIONS):
    z         = np.random.standard_normal((N_DAYS, len(TICKERS)))
    corr_z    = z @ chol.T                                    # apply correlation
    sim_ret   = mu_vec + corr_z                               # returns with drift
    port_ret  = sim_ret @ WEIGHTS                             # weighted daily return
    cum_ret   = np.cumprod(1 + port_ret)
    all_paths[sim] = cum_ret * PORTFOLIO_VALUE
    final_values[sim] = cum_ret[-1] * PORTFOLIO_VALUE

print(f"✅  {N_SIMULATIONS:,} Monte Carlo simulations complete")
print(f"💰  Expected Final Value : ${np.mean(final_values):,.0f}")
print(f"📉  5th Percentile (VaR) : ${np.percentile(final_values, 5):,.0f}")
print(f"📈  95th Percentile      : ${np.percentile(final_values, 95):,.0f}")
print(f"⚠️   Probability of Loss  : {np.mean(final_values < PORTFOLIO_VALUE):.1%}")


✅  10,000 Monte Carlo simulations complete
💰  Expected Final Value : $137,788
📉  5th Percentile (VaR) : $81,709
📈  95th Percentile      : $213,279
⚠️   Probability of Loss  : 17.1%


In [42]:
# ── Monte Carlo Fan Chart ────────────────────────────────────────────────────────
days = np.arange(1, N_DAYS + 1)

p5, p25, p50, p75, p95 = [
    np.percentile(all_paths, p, axis=0) for p in [5, 25, 50, 75, 95]
]

fig = go.Figure()

# Confidence bands
fig.add_trace(go.Scatter(x=np.concatenate([days, days[::-1]]),
    y=np.concatenate([p95, p5[::-1]]),
    fill='toself', fillcolor='rgba(52,152,219,0.08)',
    line=dict(color='rgba(0,0,0,0)'), name='5th–95th pct'))
fig.add_trace(go.Scatter(x=np.concatenate([days, days[::-1]]),
    y=np.concatenate([p75, p25[::-1]]),
    fill='toself', fillcolor='rgba(52,152,219,0.15)',
    line=dict(color='rgba(0,0,0,0)'), name='25th–75th pct'))

# Sample paths (50 random)
idx_sample = np.random.choice(N_SIMULATIONS, 50, replace=False)
for i in idx_sample:
    fig.add_trace(go.Scatter(x=days, y=all_paths[i],
        line=dict(color='rgba(255,255,255,0.04)', width=0.6),
        showlegend=False))

fig.add_trace(go.Scatter(x=days, y=p50, name='Median',
    line=dict(color='#2ecc71', width=2.5)))
fig.add_trace(go.Scatter(x=days, y=p5, name='5th Pct (VaR)',
    line=dict(color='#e74c3c', width=2, dash='dash')))
fig.add_trace(go.Scatter(x=days, y=p95, name='95th Pct',
    line=dict(color='#3498db', width=2, dash='dash')))
fig.add_hline(y=PORTFOLIO_VALUE, line_color='white', line_dash='dot',
              annotation_text='Initial $100k')

fig.update_layout(template='plotly_dark', height=500,
    title=f'Monte Carlo Simulation — {N_SIMULATIONS:,} Paths × {N_DAYS} Days',
    xaxis_title='Trading Days', yaxis_title='Portfolio Value ($)',
    hovermode='x unified')
fig.show()


In [43]:
# ── Final Value Distribution ─────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Histogram(x=final_values / 1000, nbinsx=120,
    histnorm='probability density', name='Final Values',
    marker_color='#3498db', opacity=0.7))

# KDE overlay
from scipy.stats import gaussian_kde
kde   = gaussian_kde(final_values / 1000, bw_method=0.1)
x_kde = np.linspace(final_values.min() / 1000, final_values.max() / 1000, 400)
fig.add_trace(go.Scatter(x=x_kde, y=kde(x_kde), name='KDE',
    line=dict(color='white', width=2)))

fig.add_vline(x=PORTFOLIO_VALUE / 1000, line_color='yellow', line_dash='dash',
              annotation_text='Initial $100k')
fig.add_vline(x=np.percentile(final_values, 5) / 1000, line_color='#e74c3c',
              annotation_text='5th pct')
fig.add_vline(x=np.mean(final_values) / 1000, line_color='#2ecc71',
              annotation_text='Mean')

fig.update_layout(template='plotly_dark', height=400,
    title='Distribution of Portfolio Values After 1 Year',
    xaxis_title='Portfolio Value ($k)', yaxis_title='Density')
fig.show()


## 7. Interactive Analytics Dashboard <a id='7'></a>

In [44]:
# ── Comprehensive Multi-Panel Dashboard ──────────────────────────────────────────
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[
        'Cumulative Returns',        'Annual Volatility (%)',      'Sharpe Ratio',
        'Portfolio Drawdown',        'Correlation Heatmap',        'VaR Comparison',
        'Return Distribution',       'Beta vs Return',             'Monte Carlo (sample)',
    ],
    specs=[
        [{'colspan': 1}, {}, {}],
        [{}, {'type': 'heatmap'}, {}],
        [{}, {}, {}],
    ],
    vertical_spacing=0.12, horizontal_spacing=0.08,
    row_heights=[0.35, 0.35, 0.30]
)

# 1) Cumulative returns
fig.add_trace(go.Scatter(x=cum_port.index,  y=cum_port,  name='Portfolio',
    line=dict(color='#2ecc71', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=cum_bench.index, y=cum_bench, name='SPY',
    line=dict(color='#e74c3c', width=1.5, dash='dash')), row=1, col=1)

# 2) Volatility bar
vols = daily_returns.std() * np.sqrt(252) * 100
fig.add_trace(go.Bar(x=TICKERS, y=vols.values, marker_color='#3498db',
    name='Volatility'), row=1, col=2)

# 3) Sharpe ratio
sharpes = [sharpe_ratio(daily_returns[t]) for t in TICKERS]
colors  = ['#2ecc71' if s > 1 else '#e67e22' if s > 0 else '#e74c3c' for s in sharpes]
fig.add_trace(go.Bar(x=TICKERS, y=sharpes, marker_color=colors,
    name='Sharpe'), row=1, col=3)

# 4) Drawdown
fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown, fill='tozeroy',
    fillcolor='rgba(231,76,60,0.3)', line=dict(color='#e74c3c', width=1),
    name='Drawdown'), row=2, col=1)

# 5) Heatmap
fig.add_trace(go.Heatmap(z=corr_matrix.values, x=TICKERS, y=TICKERS,
    colorscale='RdBu_r', zmin=-1, zmax=1, showscale=False,
    text=corr_matrix.round(2).values.astype(str), texttemplate='%{text}'),
    row=2, col=2)

# 6) VaR comparison
hist_vars  = [historical_var(daily_returns[t]) * 100 for t in TICKERS]
param_vars = [parametric_var(daily_returns[t]) * 100 for t in TICKERS]
cvars      = [conditional_var(daily_returns[t]) * 100 for t in TICKERS]
fig.add_trace(go.Bar(x=TICKERS, y=hist_vars,  name='Hist VaR',  marker_color='#e74c3c'), row=2, col=3)
fig.add_trace(go.Bar(x=TICKERS, y=param_vars, name='Param VaR', marker_color='#e67e22'), row=2, col=3)
fig.add_trace(go.Bar(x=TICKERS, y=cvars,      name='CVaR',      marker_color='#9b59b6'), row=2, col=3)

# 7) Return distribution
fig.add_trace(go.Histogram(x=port_daily_ret * 100, nbinsx=80,
    marker_color='#3498db', opacity=0.7, name='Port Returns'), row=3, col=1)

# 8) Beta vs Annual Return scatter
betas   = [portfolio_beta(daily_returns[t].values, bench_returns.values) for t in TICKERS]
ann_rets = (daily_returns.mean() * TRADING_DAYS * 100).values
fig.add_trace(go.Scatter(
    x=betas, y=ann_rets, mode='markers+text',
    text=TICKERS, textposition='top center',
    marker=dict(size=14, color=ann_rets, colorscale='RdYlGn', showscale=False),
    name='Beta vs Return'), row=3, col=2)

# 9) Monte Carlo sample
for i in np.random.choice(N_SIMULATIONS, 200, replace=False):
    fig.add_trace(go.Scatter(x=days, y=all_paths[i] / 1000,
        line=dict(color='rgba(52,152,219,0.06)', width=0.5),
        showlegend=False), row=3, col=3)
fig.add_trace(go.Scatter(x=days, y=p50 / 1000, name='MC Median',
    line=dict(color='#2ecc71', width=2.5)), row=3, col=3)
fig.add_trace(go.Scatter(x=days, y=p5 / 1000, name='MC 5th pct',
    line=dict(color='#e74c3c', width=2, dash='dash')), row=3, col=3)

fig.update_layout(
    template='plotly_dark', height=950,
    title=dict(text='📊 Finance & Risk Analytics — Interactive Dashboard', font=dict(size=20)),
    showlegend=False, barmode='group'
)
fig.show()
print("\n✅  Dashboard rendered. Scroll up to interact with all panels.")



✅  Dashboard rendered. Scroll up to interact with all panels.


## 📋 Executive Summary

```
KEY FINDINGS
─────────────────────────────────────────────────────────────────────
Portfolio vs Benchmark   See annual returns table (Section 2)
VaR (95%, 1-day)         Historical and Parametric VaR computed per asset
Best Sharpe              Ranking in metrics table (Section 3)
ML Directional Accuracy  Ensemble model tested on held-out data (Section 5)
Worst Drawdown           See drawdown chart (Section 3)
Monte Carlo P(Loss)      Simulated over 10,000 paths (Section 6)
─────────────────────────────────────────────────────────────────────
```

## 🛠️ Tech Stack

| Library | Use |
|---------|-----|
| `yfinance` | Market data download |
| `pandas` / `numpy` | Data wrangling & numerical computation |
| `scipy.stats` | Statistical distributions, parametric VaR |
| `scikit-learn` | Random Forest, Gradient Boosting, TimeSeriesSplit |
| `plotly` | All interactive visualizations |
| `matplotlib` / `seaborn` | Static fallback plots |

## 🔭 Potential Extensions

- **Options pricing** — Black-Scholes, implied volatility surface  
- **Factor modelling** — Fama-French 3/5-factor decomposition  
- **Portfolio optimisation** — Efficient frontier, Max Sharpe, Min Variance  
- **Deep learning** — LSTM / Transformer-based sequence prediction  
- **Streamlit deployment** — Turn this notebook into a live web app  
- **Real-time alerts** — Price drop / VaR breach notifications via email/SMS  
